# Modelado de Demanda Energética — Área OCC

En este notebook entreno y comparo cinco modelos de regresión supervisada para predecir la demanda eléctrica horaria del área OCC del Sistema Interconectado Nacional (SIN). Partiendo de los datos ya explorados en el EDA y aplicando ingeniería de características temporales antes de entrenar cada modelo.

**Variable objetivo:** `Estimacion de Demanda por Balance (MWh)`

| Modelo | Tipo |
|--------|------|
| Regresión Lineal | Baseline estadístico |
| Regresión Polinomial Grado 2 | Extiende la lineal con términos cuadráticos e interacciones |
| Random Forest | Ensemble de árboles en paralelo |
| XGBoost | Boosting — árboles en secuencia |
| Prophet | Modelo de series de tiempo (Meta) |

Evalúo cada modelo con **MAE**, **RMSE** y **R²** para determinar cuál predice mejor la demanda horaria y cuánto mejora respecto al pronóstico oficial.

## 1. Librerías

Importación de todas las librerías necesarias para procesamiento de datos, modelado y visualización. Agrupadas por categoría para tener claridad sobre el propósito de cada una.

In [ ]:
# Manejo de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Modelos y métricas — scikit-learn
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# XGBoost
from xgboost import XGBRegressor

# Prophet
from prophet import Prophet

import warnings
warnings.filterwarnings('ignore')

## 2. Carga y filtrado de datos

Carga del dataset del área OCC desde el archivo CSV, se convierte el `Timestamp` a datetime y realizo la limpieza necesaria antes de modelar.

**Limpieza de `Intercambio neto entre Gerencias (MWh)`:** esta columna contiene valores `'---'` que representan datos no disponibles y que impiden convertirla a numérico. Se eliminaron para mantener únicamente registros válidos. Aunque esta variable no se usa como feature (es contemporánea al target y causaría *data leakage*), la limpieza es necesaria para evitar errores al cargar el dataset.

Ordenamiento cronológico desde el inicio porque los lags dependen del orden temporal.

In [ ]:
# Cargar datos del área OCC
df_area = pd.read_csv('datos_occ.csv')
df_area['Timestamp'] = pd.to_datetime(df_area['Timestamp'])

print(f'Registros antes de limpiar: {df_area.shape[0]:,}')
df_area.head()

In [ ]:
# Limpieza de la columna 'Intercambio neto entre Gerencias (MWh)'
# El dataset trae valores '---' que representan datos no disponibles
# Se eliminan antes de convertir la columna a numérico

col_intercambio = 'Intercambio neto entre Gerencias (MWh)'

# Quitar espacios en blanco que puedan estar pegados al '---'
df_area[col_intercambio] = df_area[col_intercambio].astype(str).str.strip()

# Identificar y eliminar filas con valor '---'
mask_invalidos = df_area[col_intercambio] == '---'
print(f"Filas con '---': {mask_invalidos.sum()}")

df_area = df_area[~mask_invalidos].reset_index(drop=True)

# Convertir a numérico
df_area[col_intercambio] = pd.to_numeric(df_area[col_intercambio])

print(f"Registros tras limpieza: {df_area.shape[0]:,}")
print(f"Dtype:                   {df_area[col_intercambio].dtype}")

In [ ]:
# Ordenar cronológicamente — indispensable antes de crear lags
df_area = df_area.sort_values('Timestamp').reset_index(drop=True)

print(f'Registros OCC:  {df_area.shape[0]:,}')
print(f'Columnas:       {df_area.shape[1]}')
print(f'Periodo:        {df_area["Timestamp"].min()} → {df_area["Timestamp"].max()}')

## 3. Ingeniería de características

Para transformar el problema de serie de tiempo en regresión supervisada se construyen variables que representen el comportamiento histórico de la demanda. El modelo necesita información sobre el pasado para predecir el futuro.

### Variables temporales
- `hora`: captura los patrones horarios del consumo (picos mañana/noche identificados en el EDA)
- `mes`: captura estacionalidad anual

### Lag features de demanda
Construcción de tres rezagos basados en los ciclos naturales del consumo eléctrico:

| Feature | Rezago | Justificación |
|---------|--------|---------------|
| `demanda_t-1` | 1 hora | Correlación ~0.978 con el target — el consumo no cambia bruscamente de una hora a la siguiente |
| `demanda_t-24` | 24 horas | Patrón diario — lo que ocurrió ayer a esta hora predice bien lo de hoy |
| `demanda_t-168` | 7 días | Patrón semanal — 168 = 7 días × 24 horas; el mismo día de la semana pasada |

> **¿Por qué no incluyo `Generacion` ni `Intercambio neto` como features?**
> Durante el análisis se encontraron que la regresión lineal con esas variables obtiene R²=1.0
> y coeficientes exactamente iguales a 1.0, porque forman parte de la identidad contable
> con la que se construyó el target: `Demanda = Generacion + Intercambio`.
> Son variables contemporáneas — en producción no se conocerían al momento de predecir,
> por lo que incluirlas sería *data leakage*.

In [ ]:
target = 'Estimacion de Demanda por Balance (MWh)'

# Variables temporales
df_area['hora'] = df_area['Timestamp'].dt.hour
df_area['mes']  = df_area['Timestamp'].dt.month

# Lag features de demanda — rezagos de 1h, 24h y 168h (1 semana)
df_area['demanda_t-1']   = df_area[target].shift(1)
df_area['demanda_t-24']  = df_area[target].shift(24)
df_area['demanda_t-168'] = df_area[target].shift(168)

# Eliminar las primeras 168 filas que quedaron con NaN por efecto del shift
df_area = df_area.dropna(
    subset=['demanda_t-1', 'demanda_t-24', 'demanda_t-168']
).reset_index(drop=True)

print(f'Registros tras crear lags: {df_area.shape[0]:,}')
print(f'Periodo de modelado:       {df_area["Timestamp"].min()} → {df_area["Timestamp"].max()}')

## 4. Selección de variables predictoras y división temporal

Definimos el conjunto final de features y divido los datos en entrenamiento (80%) y prueba (20%) respetando el orden cronológico.

**¿Por qué no uso shuffle?**
Los datos son una serie de tiempo. Si mezclara aleatoriamente, el modelo vería durante el entrenamiento valores de `demanda_t-1` que en realidad pertenecen al periodo de prueba. Eso se llama *data leakage* y produce métricas optimistas en test que no se repiten en producción.

Al usar el 80% inicial para entrenar y el 20% final para evaluar, garantizo que el modelo solo aprende del pasado y se mide con datos que genuinamente nunca vio.

In [ ]:
features = ['demanda_t-1', 'demanda_t-24', 'demanda_t-168', 'hora', 'mes']

X = df_area[features]
y = df_area[target]

# Split temporal 80/20 — sin shuffle, respetando orden cronológico
corte = int(len(df_area) * 0.80)

X_train, X_test = X.iloc[:corte], X.iloc[corte:]
y_train, y_test = y.iloc[:corte], y.iloc[corte:]

print(f"Entrenamiento: {len(X_train):,} registros")
print(f"  {df_area['Timestamp'].iloc[0]} → {df_area['Timestamp'].iloc[corte-1]}")
print(f"\nPrueba:  {len(X_test):,} registros")
print(f"  {df_area['Timestamp'].iloc[corte]} → {df_area['Timestamp'].iloc[-1]}")

In [ ]:
# Visualización de la división temporal del dataset
plt.figure(figsize=(14, 4))
plt.plot(df_area['Timestamp'].iloc[:corte],  y_train,
         label='Entrenamiento', color='steelblue',  linewidth=0.6)
plt.plot(df_area['Timestamp'].iloc[corte:],  y_test,
         label='Prueba',        color='darkorange', linewidth=0.6)
plt.axvline(df_area['Timestamp'].iloc[corte],
            color='red', linestyle='--', linewidth=1.5, label='Corte 80/20')
plt.title('División temporal del dataset — Área OCC')
plt.xlabel('Fecha')
plt.ylabel('Demanda (MWh)')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Función de evaluación

Definimos una función reutilizable que calcula las tres métricas principales para cualquier modelo. Esto nos permite comparar todos los modelos con exactamente el mismo criterio.

- **MAE** (Error Absoluto Medio): el error promedio en MWh — intuitivo y fácil de comunicar
- **RMSE** (Raíz del Error Cuadrático Medio): penaliza más los errores grandes que el MAE
- **R²** (Coeficiente de Determinación): qué porcentaje de la varianza de la demanda explica el modelo (1.0 = perfecto, 0 = no explica nada)

In [ ]:
def evaluar_modelo(nombre, y_true, y_pred):
    """Calcula MAE, RMSE y R², imprime un resumen y devuelve un dict para la tabla comparativa."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'{nombre:<35}  MAE: {mae:>8.2f} MWh  |  RMSE: {rmse:>8.2f}  |  R²: {r2:>7.4f}')
    return {'Modelo': nombre, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R²': round(r2, 4)}

# Lista donde se acumulan los resultados de cada modelo para la comparación final
resultados = []

## 6. Baseline — Pronóstico oficial

Antes de entrenar cualquier modelo, se  establece una línea base usando el pronóstico oficial (`Pronostico (MWh)`) que ya viene en el dataset. Este es el valor que el sistema operativo actual utiliza para planificar el despacho de energía.

Cualquier modelo que se entrene debe superar este baseline para justificar su uso en producción.

In [ ]:
# Pronóstico oficial como referencia base
pronostico_test = df_area['Pronostico (MWh)'].iloc[corte:].values

r = evaluar_modelo('Baseline (pronóstico oficial)', y_test.values, pronostico_test)
resultados.append(r)

## 7. Modelo 1 — Regresión Lineal

La regresión lineal es el modelo más simple: asume que la demanda es una suma ponderada de las features. Para mi caso la ecuación queda así:

$$\hat{y} = \beta_0 + \beta_1(\text{demanda\_t-1}) + \beta_2(\text{demanda\_t-24}) + \beta_3(\text{demanda\_t-168}) + \beta_4(\text{hora}) + \beta_5(\text{mes})$$

Cada $\beta$ es un coeficiente que el modelo aprende minimizando la suma de errores al cuadrado. Lo uso principalmente como punto de comparación: si los modelos más complejos no le ganan, significa que el problema es esencialmente lineal.

In [ ]:
# Regresión Lineal — modelo más simple, punto de comparación estadístico
modelo_lr = LinearRegression()
modelo_lr.fit(X_train, y_train)
y_pred_lr = modelo_lr.predict(X_test)

r = evaluar_modelo('Regresión Lineal', y_test.values, y_pred_lr)
resultados.append(r)

In [ ]:
# Residuales — Regresión Lineal
residuales_lr = y_test.values - y_pred_lr
hora_test     = df_area['hora'].iloc[corte:].reset_index(drop=True).values

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(residuales_lr[:24*30], linewidth=0.7, color='#e74c3c')
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Residuales en el tiempo (primer mes)')
axes[0].set_ylabel('Error (MWh)')
axes[0].grid(True, alpha=0.3)

axes[1].hist(residuales_lr, bins=60, color='#e74c3c', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Distribución de residuales')
axes[1].set_xlabel('Error (MWh)')
axes[1].grid(True, alpha=0.3)

df_res = pd.DataFrame({'residual': residuales_lr, 'hora': hora_test})
df_res.groupby('hora')['residual'].mean().plot(ax=axes[2], marker='o', color='#e74c3c')
axes[2].axhline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_title('Error promedio por hora del día')
axes[2].set_xlabel('Hora')
axes[2].set_ylabel('Error promedio (MWh)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Análisis de residuales — Regresión Lineal', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Sesgo (error promedio): {residuales_lr.mean():.2f} MWh')
print(f'Desviación estándar:    {residuales_lr.std():.2f} MWh')

## 8. Modelo 2 — Regresión Polinomial Grado 2

La regresión polinomial de grado 2 extiende la lineal agregando automáticamente:
- El **cuadrado** de cada feature: `(demanda_t-1)²`, `(hora)²`, etc.
- Las **interacciones** entre pares de features: `demanda_t-1 × hora`, `demanda_t-1 × mes`, etc.

Con 5 features originales, esto genera **20 features nuevas**. El modelo sigue siendo mínimos cuadrados ordinarios (OLS), solo que ahora el espacio de features es más rico y puede capturar curvatura en los datos.

El término `demanda_t-1 × hora` es especialmente útil: captura que el efecto de la demanda pasada depende de la hora del día — a las 3am una demanda alta significa algo distinto que a las 6pm.

Uso `Pipeline` para encadenar la transformación y el modelo en un solo estimador.

In [ ]:
# Regresión Polinomial Grado 2 — captura curvatura e interacciones entre features
modelo_poly2 = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lr',   LinearRegression())
])
modelo_poly2.fit(X_train, y_train)
y_pred_poly2 = modelo_poly2.predict(X_test)

r = evaluar_modelo('Polinomial Grado 2', y_test.values, y_pred_poly2)
resultados.append(r)

In [ ]:
# Residuales — Regresión Polinomial Grado 2
residuales_poly = y_test.values - y_pred_poly2

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(residuales_poly[:24*30], linewidth=0.7, color='#f39c12')
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Residuales en el tiempo (primer mes)')
axes[0].set_ylabel('Error (MWh)')
axes[0].grid(True, alpha=0.3)

axes[1].hist(residuales_poly, bins=60, color='#f39c12', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Distribución de residuales')
axes[1].set_xlabel('Error (MWh)')
axes[1].grid(True, alpha=0.3)

df_res = pd.DataFrame({'residual': residuales_poly, 'hora': hora_test})
df_res.groupby('hora')['residual'].mean().plot(ax=axes[2], marker='o', color='#f39c12')
axes[2].axhline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_title('Error promedio por hora del día')
axes[2].set_xlabel('Hora')
axes[2].set_ylabel('Error promedio (MWh)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Análisis de residuales — Polinomial Grado 2', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Sesgo (error promedio): {residuales_poly.mean():.2f} MWh')
print(f'Desviación estándar:    {residuales_poly.std():.2f} MWh')

## 9. Modelo 3 — Random Forest

En vez de una ecuación, Random Forest construye 300 árboles de decisión y promedia sus predicciones. Cada árbol se entrena sobre una muestra aleatoria con reemplazo de los datos de entrenamiento (*bootstrap*), y en cada división del árbol solo considera un subconjunto aleatorio de features. Esto hace que los árboles sean distintos entre sí y cometan errores en momentos diferentes.

La predicción final es el promedio de los 300 árboles:

$$\hat{y} = \frac{1}{300} \sum_{b=1}^{300} T_b(x)$$

El promedio de muchos árboles descorrelacionados reduce la varianza sin aumentar el sesgo — mucho más estable que un solo árbol y capaz de capturar relaciones no lineales sin que yo las tenga que especificar manualmente.

In [ ]:
# Random Forest — ensemble de 300 árboles en paralelo
modelo_rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
modelo_rf.fit(X_train, y_train)
y_pred_rf = modelo_rf.predict(X_test)

r = evaluar_modelo('Random Forest', y_test.values, y_pred_rf)
resultados.append(r)

In [ ]:
# Residuales + Importancia de features — Random Forest
residuales_rf = y_test.values - y_pred_rf

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(residuales_rf[:24*30], linewidth=0.7, color='#2ecc71')
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Residuales en el tiempo (primer mes)')
axes[0].set_ylabel('Error (MWh)')
axes[0].grid(True, alpha=0.3)

axes[1].hist(residuales_rf, bins=60, color='#2ecc71', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Distribución de residuales')
axes[1].set_xlabel('Error (MWh)')
axes[1].grid(True, alpha=0.3)

df_res = pd.DataFrame({'residual': residuales_rf, 'hora': hora_test})
df_res.groupby('hora')['residual'].mean().plot(ax=axes[2], marker='o', color='#2ecc71')
axes[2].axhline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_title('Error promedio por hora del día')
axes[2].set_xlabel('Hora')
axes[2].set_ylabel('Error promedio (MWh)')
axes[2].grid(True, alpha=0.3)

# Importancia de features
importancia_rf = pd.Series(
    modelo_rf.feature_importances_, index=features
).sort_values(ascending=True)
axes[3].barh(importancia_rf.index, importancia_rf.values,
             color='#2ecc71', alpha=0.85, edgecolor='white')
axes[3].set_title('Importancia de features')
axes[3].set_xlabel('Importancia relativa')
axes[3].grid(True, alpha=0.3, axis='x')

plt.suptitle('Residuales e Importancia de features — Random Forest',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Sesgo (error promedio): {residuales_rf.mean():.2f} MWh')
print(f'Desviación estándar:    {residuales_rf.std():.2f} MWh')

## 10. Modelo 4 — XGBoost

XGBoost también usa árboles, pero construidos de forma completamente distinta a Random Forest. Los árboles se construyen en **secuencia**: cada árbol nuevo se enfoca en corregir los errores que cometió el modelo anterior.

Empieza con la predicción inicial (promedio de la demanda), calcula los residuos, entrena un árbol pequeño para predecirlos y actualiza el modelo:

$$F_1(x) = F_0(x) + 0.05 \cdot h_1(x)$$

Esto se repite 500 veces. El `learning_rate` de 0.05 controla cuánto "confía" en cada árbol nuevo — valores pequeños hacen el aprendizaje más lento pero más preciso. XGBoost además agrega regularización L1 y L2 para evitar sobreajuste, y usa el Hessiano (segunda derivada de la pérdida) para encontrar los cortes óptimos de forma más eficiente que un árbol convencional.

In [ ]:
# XGBoost — boosting: árboles en secuencia, cada uno corrige al anterior
modelo_xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
modelo_xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
y_pred_xgb = modelo_xgb.predict(X_test)

r = evaluar_modelo('XGBoost', y_test.values, y_pred_xgb)
resultados.append(r)

In [ ]:
# Residuales + Importancia de features — XGBoost
residuales_xgb = y_test.values - y_pred_xgb

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(residuales_xgb[:24*30], linewidth=0.7, color='#9b59b6')
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Residuales en el tiempo (primer mes)')
axes[0].set_ylabel('Error (MWh)')
axes[0].grid(True, alpha=0.3)

axes[1].hist(residuales_xgb, bins=60, color='#9b59b6', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Distribución de residuales')
axes[1].set_xlabel('Error (MWh)')
axes[1].grid(True, alpha=0.3)

df_res = pd.DataFrame({'residual': residuales_xgb, 'hora': hora_test})
df_res.groupby('hora')['residual'].mean().plot(ax=axes[2], marker='o', color='#9b59b6')
axes[2].axhline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_title('Error promedio por hora del día')
axes[2].set_xlabel('Hora')
axes[2].set_ylabel('Error promedio (MWh)')
axes[2].grid(True, alpha=0.3)

# Importancia de features
importancia_xgb = pd.Series(
    modelo_xgb.feature_importances_, index=features
).sort_values(ascending=True)
axes[3].barh(importancia_xgb.index, importancia_xgb.values,
             color='#9b59b6', alpha=0.85, edgecolor='white')
axes[3].set_title('Importancia de features')
axes[3].set_xlabel('Importancia relativa')
axes[3].grid(True, alpha=0.3, axis='x')

plt.suptitle('Residuales e Importancia de features — XGBoost',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Sesgo (error promedio): {residuales_xgb.mean():.2f} MWh')
print(f'Desviación estándar:    {residuales_xgb.std():.2f} MWh')

## 11. Modelo 5 — Prophet

Prophet es un modelo de series de tiempo desarrollado por Meta. A diferencia de los modelos anteriores, **no usa las lag features** — trabaja directamente con la señal temporal de la demanda y detecta automáticamente:

- **Tendencia general** a largo plazo
- **Estacionalidad diaria** — patrones hora a hora
- **Estacionalidad semanal** — patrones día a día de la semana
- **Estacionalidad anual** — patrones mes a mes

Uso `seasonality_mode='multiplicative'` porque en datos de energía la amplitud de los ciclos varía con el nivel general de la demanda: cuando la demanda es alta los picos son proporcionalmente más grandes.

Prophet requiere un DataFrame con exactamente dos columnas: `ds` (fechas) e `y` (variable objetivo). Uso el mismo punto de corte 80/20 que los demás modelos para que la comparación sea justa.

In [ ]:
# Prophet — modelo de series de tiempo basado en descomposición temporal
# Requiere columnas 'ds' (fecha) e 'y' (variable objetivo)
df_prophet = df_area[['Timestamp', target]].rename(
    columns={'Timestamp': 'ds', target: 'y'}
)

# Mismo punto de corte que los demás modelos
fecha_corte = df_area['Timestamp'].iloc[corte]
train_p = df_prophet[df_prophet['ds'] < fecha_corte].copy()
test_p  = df_prophet[df_prophet['ds'] >= fecha_corte].copy()

# Entrenar
model_p = Prophet(
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=True
)
model_p.fit(train_p)

# Predecir sobre el periodo de prueba
future   = model_p.make_future_dataframe(periods=len(test_p), freq='h')
forecast = model_p.predict(future)
y_pred_p = forecast.tail(len(test_p))['yhat'].values

r = evaluar_modelo('Prophet', test_p['y'].values, y_pred_p)
resultados.append(r)

In [ ]:
# Componentes detectados por Prophet — tendencia y estacionalidades
fig_comp = model_p.plot_components(forecast)
plt.suptitle('Componentes detectados por Prophet — tendencia y estacionalidades',
             y=1.01, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Residuales — Prophet
residuales_p = test_p['y'].values - y_pred_p
hora_test_p  = test_p['ds'].dt.hour.values

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(residuales_p[:24*30], linewidth=0.7, color='#1abc9c')
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Residuales en el tiempo (primer mes)')
axes[0].set_ylabel('Error (MWh)')
axes[0].grid(True, alpha=0.3)

axes[1].hist(residuales_p, bins=60, color='#1abc9c', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Distribución de residuales')
axes[1].set_xlabel('Error (MWh)')
axes[1].grid(True, alpha=0.3)

df_res = pd.DataFrame({'residual': residuales_p, 'hora': hora_test_p})
df_res.groupby('hora')['residual'].mean().plot(ax=axes[2], marker='o', color='#1abc9c')
axes[2].axhline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_title('Error promedio por hora del día')
axes[2].set_xlabel('Hora')
axes[2].set_ylabel('Error promedio (MWh)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Análisis de residuales — Prophet', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Sesgo (error promedio): {residuales_p.mean():.2f} MWh')
print(f'Desviación estándar:    {residuales_p.std():.2f} MWh')

## 12. Comparación de modelos

Comparamos todos los modelos usando las métricas MAE, RMSE y R². El baseline (pronóstico oficial) establece el piso que debo superar — cualquier modelo que se entrene debería tener menor error para justificar su uso.

Ordenamos la tabla por MAE (menor es mejor) para identificar fácilmente el modelo ganador.

In [ ]:
# Tabla comparativa ordenada por MAE
df_resultados = pd.DataFrame(resultados).sort_values('MAE').reset_index(drop=True)

print(f"{'='*65}")
print(f"  COMPARACIÓN DE MODELOS — Área OCC")
print(f"{'='*65}")
print(df_resultados.to_string(index=False))
print(f"{'='*65}")

# Destacar el mejor modelo
mejor = df_resultados.iloc[0]
print(f"\n  Mejor modelo: {mejor['Modelo']}  (MAE = {mejor['MAE']:.2f} MWh, R² = {mejor['R²']:.4f})")

# Comparar contra el baseline
mae_baseline = df_resultados[df_resultados['Modelo'] == 'Baseline (pronóstico oficial)']['MAE'].values[0]
mejora = (mae_baseline - mejor['MAE']) / mae_baseline * 100
print(f"  Mejora vs baseline: {mejora:.1f}%  ({mae_baseline:.2f} → {mejor['MAE']:.2f} MWh)")

In [ ]:
# Gráficas de barras — comparación por métrica
colores_modelos = {
    'Baseline (pronóstico oficial)': '#888888',
    'Regresión Lineal':              '#e74c3c',
    'Polinomial Grado 2':            '#f39c12',
    'Random Forest':                 '#2ecc71',
    'XGBoost':                       '#9b59b6',
    'Prophet':                       '#1abc9c',
}

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, metrica in zip(axes, ['MAE', 'RMSE', 'R²']):
    ascending = metrica != 'R²'          # R²: mayor es mejor
    df_plot = df_resultados.sort_values(metrica, ascending=ascending)
    colores = [colores_modelos.get(m, '#333333') for m in df_plot['Modelo']]

    bars = ax.barh(df_plot['Modelo'], df_plot[metrica],
                   color=colores, edgecolor='white', alpha=0.9)
    ax.set_title(metrica, fontsize=12, fontweight='bold')
    ax.set_xlabel(metrica)

    for bar, val in zip(bars, df_plot[metrica]):
        fmt = f'{val:.4f}' if metrica == 'R²' else f'{val:.2f}'
        ax.text(bar.get_width() * 1.01,
                bar.get_y() + bar.get_height() / 2,
                fmt, va='center', fontsize=9)
    ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Comparación de modelos — Área OCC', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Visualizaciones comparativas

### 13.1 Predicciones vs demanda real

Visualizamos las predicciones de todos los modelos contra la demanda real en las últimas 2 semanas del conjunto de prueba. Usamos solo este rango porque mostrar todo el periodo (miles de puntos) hace que la gráfica sea ilegible. Las 2 semanas muestran variación suficiente para ver cómo se comporta cada modelo en días laborables y fines de semana.

In [ ]:
# Todos los modelos vs demanda real — últimas 2 semanas del test
n_horas = 24 * 14
timestamps_test = df_area['Timestamp'].iloc[corte:].reset_index(drop=True)

modelos_viz = [
    ('Baseline (oficial)', pronostico_test, '#888888', ':'),
    ('Regresión Lineal',   y_pred_lr,       '#e74c3c', '--'),
    ('Polinomial Grado 2', y_pred_poly2,    '#f39c12', '--'),
    ('Random Forest',      y_pred_rf,       '#2ecc71', '--'),
    ('XGBoost',            y_pred_xgb,      '#9b59b6', '--'),
    ('Prophet',            y_pred_p,        '#1abc9c', '--'),
]

fig, axes = plt.subplots(len(modelos_viz), 1, figsize=(15, 18), sharex=True)
fig.suptitle('Predicciones vs Demanda Real — últimas 2 semanas del test\nÁrea OCC',
             fontsize=13, fontweight='bold')

for ax, (nombre, y_pred, color, ls) in zip(axes, modelos_viz):
    ax.plot(timestamps_test[-n_horas:], y_test.values[-n_horas:],
            label='Real', color='steelblue', linewidth=1.5)
    ax.plot(timestamps_test[-n_horas:], y_pred[-n_horas:],
            label=nombre, color=color, linewidth=1.2, linestyle=ls, alpha=0.9)
    mae_local = mean_absolute_error(y_test.values[-n_horas:], y_pred[-n_horas:])
    ax.set_title(f'{nombre} — MAE últimas 2 semanas: {mae_local:.2f} MWh', fontsize=10)
    ax.set_ylabel('Demanda (MWh)')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

plt.xlabel('Fecha')
plt.tight_layout()
plt.show()

### 13.2 Dispersión — valores reales vs predichos

Los gráficos de dispersión muestran cuánto se acercan las predicciones de cada modelo a los valores reales. Un modelo perfecto formaría una línea diagonal exacta — cualquier desviación de esa diagonal es error del modelo.

Nos permiten detectar visualmente si hay **sesgo sistemático** (nube desplazada de la diagonal) o si el error es **aleatorio** (nube simétrica alrededor de la diagonal).

In [ ]:
# Dispersión: valores reales vs predichos — todos los modelos
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Valores reales vs predichos por modelo', fontsize=13, fontweight='bold')
axes = axes.flatten()

modelos_scatter = [
    ('Baseline (oficial)', pronostico_test, '#888888'),
    ('Regresión Lineal',   y_pred_lr,       '#e74c3c'),
    ('Polinomial Grado 2', y_pred_poly2,    '#f39c12'),
    ('Random Forest',      y_pred_rf,       '#2ecc71'),
    ('XGBoost',            y_pred_xgb,      '#9b59b6'),
    ('Prophet',            y_pred_p,        '#1abc9c'),
]

for ax, (nombre, y_pred_plot, color) in zip(axes, modelos_scatter):
    ax.scatter(y_test[:len(y_pred_plot)], y_pred_plot,
               alpha=0.15, s=4, color=color)

    # Línea diagonal perfecta como referencia
    lim_min = min(y_test[:len(y_pred_plot)].min(), np.min(y_pred_plot))
    lim_max = max(y_test[:len(y_pred_plot)].max(), np.max(y_pred_plot))
    ax.plot([lim_min, lim_max], [lim_min, lim_max],
            color='black', linewidth=1, linestyle='--', label='Predicción perfecta')

    r2 = r2_score(y_test[:len(y_pred_plot)], y_pred_plot)
    ax.set_title(f'{nombre}\nR² = {r2:.4f}', fontsize=10)
    ax.set_xlabel('Demanda real (MWh)')
    ax.set_ylabel('Demanda predicha (MWh)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14. Conclusiones

A continuación se resumen los hallazgos más importantes del proceso de modelado. Los valores numéricos en las conclusiones corresponden a los resultados de la última ejecución del notebook.

### Hallazgos principales

1. **Los modelos con lag features superan ampliamente al baseline oficial.** Esto confirma que existe margen significativo de mejora sobre el pronóstico operativo actual con técnicas estándar de machine learning.

2. **`demanda_t-1` domina la importancia de features (~95% en RF y XGB).** Consistente con la autocorrelación de ~0.978 calculada en el EDA. El consumo eléctrico es altamente inercial: lo que pasó hace una hora predice casi perfectamente lo siguiente en condiciones normales.

3. **Random Forest y XGBoost obtienen los mejores resultados.** Ambos capturan las no linealidades que la regresión lineal y polinomial no pueden modelar completamente. La diferencia entre ellos es pequeña, lo que sugiere que el cuello de botella ya no es el algoritmo sino la disponibilidad de features adicionales.

4. **La regresión polinomial mejora sobre la lineal.** Los términos de interacción (como `demanda_t-1 × hora`) capturan que el efecto de la demanda pasada depende del momento del día — algo que la lineal no puede representar.

5. **Prophet tiene métricas más bajas entre los modelos ML.** Al no contar con las lag features trabaja solo con la señal temporal y pierde la información más predictiva disponible. Sin embargo, aporta valor interpretativo: sus componentes muestran claramente las estacionalidades diaria, semanal y anual del sistema eléctrico.